# Tutorial 5: PyMC Surrogate Learning (`pymc_gp`)

Estimated time: 30-50 minutes

## Prerequisites
`pymc` and `arviz` installed.

## Learning aims
- Primary package aim: fit/evaluate surrogate models through the CLI and inspect artifacts
- Secondary scientific aim: build intuition for prior, posterior, and posterior predictive uncertainty

## Success criteria
- you can train a PyMC surrogate, evaluate new inputs, and interpret uncertainty width


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Ensure training dataset exists


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Locates the repo root, puts src/ on sys.path, and defines helpers that run
# the `mm` CLI in-process (run_mm_cli) and auxiliary tools like pytest/ruff
# via the active interpreter (run_tool). No shell cells, no PYTHONPATH prefix.
import io
import os
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Could not locate project root (expected src/).")

src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Run the whole notebook from the repo root so CLI artifacts and any
# CWD-relative registry lookups (e.g. eval_surrogate) resolve consistently.
os.chdir(root)

# Jupyter caches imported modules; clear bayesian_metamodeling so re-runs pick
# up the current local source.
for module_name in list(sys.modules):
    if module_name == "bayesian_metamodeling" or module_name.startswith("bayesian_metamodeling."):
        del sys.modules[module_name]

from bayesian_metamodeling.cli.main import main as mm_main


@contextmanager
def in_project_root():
    previous = Path.cwd()
    os.chdir(root)
    try:
        yield
    finally:
        os.chdir(previous)


def run_mm_cli(*args: str, check: bool = True) -> int:
    """Run `mm <args>` in-process; cross-platform, no shell."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    previous_argv = sys.argv[:]
    try:
        sys.argv = ["mm", *args]
        with in_project_root(), redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
            exit_code = mm_main()
    finally:
        sys.argv = previous_argv

    print("$ mm", " ".join(args))
    out = stdout_buf.getvalue().strip()
    err = stderr_buf.getvalue().strip()
    if out:
        print(out)
    if err:
        print(err)
    if check and exit_code != 0:
        raise RuntimeError(f"CLI command failed ({exit_code}): mm {' '.join(args)}")
    return exit_code


def run_tool(*args: str, check: bool = True) -> int:
    """Run an auxiliary tool (pytest, ruff) via the active interpreter, cross-platform."""
    cmd = list(args)
    if cmd and cmd[0] in {"pytest", "ruff"}:
        cmd = [sys.executable, "-m", *cmd]
    print("$", " ".join(args))
    with in_project_root():
        result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Tool failed ({result.returncode}): {' '.join(args)}")
    return result.returncode


In [ ]:
run_mm_cli('run', 'tutorials/specs/model.toy.grid.json')


## Step 2: Fit surrogate and list artifacts


In [ ]:
run_mm_cli('surrogate', 'fit', 'tutorials/specs/surrogate.toy.pymc_gp.json')
run_mm_cli('surrogate', 'list')


## Step 3: Evaluate on new inputs


In [ ]:
run_mm_cli('surrogate', 'eval', 'tutorials/specs/surrogate.toy.pymc_gp.json', '--inputs', '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}', '--n', '200')


## Step 4: Plot predictive mean and uncertainty (graphic)


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from bayesian_metamodeling.spec import SurrogateSpec
from bayesian_metamodeling.surrogates import eval_surrogate

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

spec_payload = json.loads((root / 'tutorials/specs/surrogate.toy.pymc_gp.json').read_text())
spec = SurrogateSpec.model_validate(spec_payload)
inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)

# summary['mean'] / ['std'] are dicts keyed by output name (multi-output aware).
out_name = spec.outputs[0]
mean = np.asarray(result['summary']['mean'][out_name], dtype=float)
std_dict = result['summary'].get('std', {})
std = np.asarray(std_dict.get(out_name, [0.0] * len(mean)), dtype=float)
x = np.arange(len(mean))

plt.figure(figsize=(6, 4))
plt.errorbar(x, mean, yerr=std, fmt='o-', capsize=4)
plt.title('PyMC surrogate predictive mean ± std')
plt.xlabel('query point index')
plt.ylabel('predicted y')
plt.grid(True, alpha=0.3)
plt.show()

## Scientific mini-lesson
- Prior: beliefs before data.
- Posterior: updated beliefs after data.
- Posterior predictive: uncertainty-aware output prediction.

Interpretation prompt:
- Where are uncertainties widest, and what does that say about data support in that region?


In [ ]:
run_tool('pytest', '-q', 'tests/test_surrogate_backends.py', '-k', 'pymc_gp_backend_fit_sample_and_logprob')
